In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
path = '/content/drive/MyDrive/MIMIC_ICU/'

In [ ]:
import pandas as pd

In [13]:
import pandas as pd

# Path to your Drive folder
path = '/content/drive/MyDrive/MIMIC-demo/'

# 1. Load Data
patients = pd.read_csv(path + 'patients.csv.gz', compression='infer')
admissions = pd.read_csv(path + 'admissions.csv.gz', compression='infer')
diagnoses = pd.read_csv(path + 'diagnoses_icd.csv.gz', compression='infer')
d_icd = pd.read_csv(path + 'd_icd_diagnoses.csv.gz', compression='infer')
chartevents = pd.read_csv(path + 'chartevents.csv.gz',
                          usecols=['subject_id', 'hadm_id', 'itemid', 'valuenum'],
                          compression='infer')

# 2. Process Vitals (HR, BP, Resp, O2)
vitals_map = {
    220045: 'HeartRate',
    220179: 'SystolicBP',
    220210: 'RespRate',
    220277: 'OxygenSat'
}
vitals = chartevents[chartevents['itemid'].isin(vitals_map.keys())].copy()
vitals['label'] = vitals['itemid'].map(vitals_map)
vitals_summary = vitals.groupby(['subject_id', 'hadm_id', 'label'])['valuenum'].mean().unstack()
vitals_summary.columns = [f"{col}_mean" for col in vitals_summary.columns]
vitals_summary = vitals_summary.reset_index()

# 3. Process Comorbidities (Diabetes, Renal Failure, Hypertension)
heart_keywords = {
    'hypertension': ['401', 'I10'],
    'diabetes': ['250', 'E11'],
    'renal_failure': ['584', 'N17']
}
comorbidity_df = admissions[['subject_id', 'hadm_id']].copy()
for disease, codes in heart_keywords.items():
    mask = diagnoses['icd_code'].str.startswith(tuple(codes), na=False)
    disease_ids = diagnoses[mask]['hadm_id'].unique()
    comorbidity_df[disease] = comorbidity_df['hadm_id'].isin(disease_ids).astype(int)

# 4. Process Primary Diagnosis (Long Title)
primary_diag = diagnoses[diagnoses['seq_num'] == 1]
primary_diag = pd.merge(primary_diag, d_icd, on=['icd_code', 'icd_version'])

# 5. MEGA JOIN: Combine all features into one Dataframe
# Step A: Demographics + Admission/Discharge Times
df = pd.merge(patients[['subject_id', 'gender', 'anchor_age']],
              admissions[['subject_id', 'hadm_id', 'admittime', 'dischtime']],
              on='subject_id')

# Step B: Add Comorbidities
df = pd.merge(df, comorbidity_df, on=['subject_id', 'hadm_id'])

# Step C: Add Vitals
df = pd.merge(df, vitals_summary, on=['subject_id', 'hadm_id'], how='left')

# Step D: Add Long Title
df = pd.merge(df, primary_diag[['hadm_id', 'long_title']], on='hadm_id', how='left')

# 6. Calculate Total LOS Days
df['admittime'] = pd.to_datetime(df['admittime'])
df['dischtime'] = pd.to_datetime(df['dischtime'])
df['total_los_days'] = (df['dischtime'] - df['admittime']).dt.total_seconds() / 86400

# 7. Final Column Selection and Ordering
final_columns = [
    'subject_id', 'gender', 'anchor_age', 'hadm_id',
    'diabetes', 'renal_failure', 'hypertension',
    'HeartRate_mean', 'OxygenSat_mean', 'RespRate_mean', 'SystolicBP_mean',
    'long_title', 'total_los_days'
]

# Ensure we only keep the requested columns
df_final = df[final_columns]

print(f"Master Dataframe created with {df_final.shape[1]} features.")
df_final.head()

Master Dataframe created with 13 features.


,subject_id,gender,anchor_age,hadm_id,diabetes,renal_failure,hypertension,HeartRate_mean,OxygenSat_mean,RespRate_mean,SystolicBP_mean,long_title,total_los_days
0,10014729,F,21,23300884,0,0,0,NaN,NaN,NaN,NaN,Other postoperative infection,8.860417
1,10014729,F,21,28889419,0,0,0,91.453125,97.446154,16.483871,100.000000,Injury to thoracic aorta,7.298611
2,10003400,F,72,23559586,0,1,0,103.488323,98.287671,19.727079,95.512195,Disruption of external operation (surgical) wo...,29.706944
3,10003400,F,72,20214994,0,1,0,112.991892,97.413374,19.551813,101.209770,"Malignant neoplasm of anus, unspecified site",23.239583
4,10003400,F,72,27296885,0,0,0,NaN,NaN,NaN,NaN,Other encephalopathy,2.809028


In [14]:
# 1. Load ICU Core and Dictionary
icu_stays = pd.read_csv(path + 'icustays.csv.gz')
d_items = pd.read_csv(path + 'd_items.csv.gz')

# 2. Load the Event Files (The ones from your screenshots)
# We aggregate these so the table doesn't get too massive to open
chartevents = pd.read_csv(path + 'chartevents.csv.gz', usecols=['stay_id', 'itemid', 'valuenum'])
inputevents = pd.read_csv(path + 'inputevents.csv.gz', usecols=['stay_id', 'amount'])
outputevents = pd.read_csv(path + 'outputevents.csv.gz', usecols=['stay_id', 'value'])

# 3. Aggregating ICU Events (Summary per stay)
# This gives us: Total fluids in, Total fluids out, and Mean vitals
char_sum = chartevents.groupby('stay_id')['valuenum'].mean().reset_index().rename(columns={'valuenum': 'avg_vitals'})
in_sum = inputevents.groupby('stay_id')['amount'].sum().reset_index().rename(columns={'amount': 'total_input_ml'})
out_sum = outputevents.groupby('stay_id')['value'].sum().reset_index().rename(columns={'value': 'total_output_ml'})

# 4. THE MEGA LINK (Joining everything into one view)
icu_master = pd.merge(icu_stays, char_sum, on='stay_id', how='left')
icu_master = pd.merge(icu_master, in_sum, on='stay_id', how='left')
icu_master = pd.merge(icu_master, out_sum, on='stay_id', how='left')

# 5. Link back to your Hospital Master (from previous code)
# This connects ICU data to Age, Gender, and the Heart Disease flags
final_mega_sheet = pd.merge(df_final, icu_master, on=['subject_id', 'hadm_id'], how='left')

print(f"Final Mega Sheet created with {final_mega_sheet.shape[1]} columns.")
final_mega_sheet.head()

Final Mega Sheet created with 22 columns.


,subject_id,gender,anchor_age,hadm_id,diabetes,renal_failure,hypertension,HeartRate_mean,OxygenSat_mean,RespRate_mean,...,total_los_days,stay_id,first_careunit,last_careunit,intime,outtime,los,avg_vitals,total_input_ml,total_output_ml
0,10014729,F,21,23300884,0,0,0,NaN,NaN,NaN,...,8.860417,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10014729,F,21,28889419,0,0,0,91.453125,97.446154,16.483871,...,7.298611,33558396.0,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2125-02-27 10:03:08,2125-03-01 21:21:37,2.471169,44.939757,24304.741086,8082.0
2,10003400,F,72,23559586,0,1,0,103.488323,98.287671,19.727079,...,29.706944,38383343.0,Coronary Care Unit (CCU),Medical Intensive Care Unit (MICU),2137-08-17 17:36:37,2137-09-02 19:17:11,16.069838,56.282463,61972.119400,150950.0
3,10003400,F,72,23559586,0,1,0,103.488323,98.287671,19.727079,...,29.706944,34577403.0,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2137-08-10 19:54:51,2137-08-13 17:54:54,2.916701,55.522472,12572.111050,2322.0
4,10003400,F,72,20214994,0,1,0,112.991892,97.413374,19.551813,...,23.239583,32128372.0,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2137-02-25 23:37:19,2137-03-10 21:29:36,12.911308,51.725388,44208.591543,29284.0


In [15]:
import pandas as pd

# Path to your dedicated ICU folder
path = '/content/drive/MyDrive/MIMIC_ICU/'

# 1. Load all files from your screenshots
# Note: compression='infer' handles .csv.gz automatically
icu_stays = pd.read_csv(path + 'icustays.csv.gz', compression='infer')
chartevents = pd.read_csv(path + 'chartevents.csv.gz', compression='infer')
inputevents = pd.read_csv(path + 'inputevents.csv.gz', compression='infer')
outputevents = pd.read_csv(path + 'outputevents.csv.gz', compression='infer')
procedureevents = pd.read_csv(path + 'procedureevents.csv.gz', compression='infer')
datetimeevents = pd.read_csv(path + 'datetimeevents.csv.gz', compression='infer')
ingredientevents = pd.read_csv(path + 'ingredientevents.csv.gz', compression='infer')

# 2. Summarize the event tables so we don't create billions of rows
# We use 'stay_id' as the primary key
vitals_sum = chartevents.groupby('stay_id')['valuenum'].agg(['mean', 'max']).reset_index()
vitals_sum.columns = ['stay_id', 'icu_vitals_mean', 'icu_vitals_max']

in_sum = inputevents.groupby('stay_id')['amount'].sum().reset_index().rename(columns={'amount': 'total_input_ml'})
out_sum = outputevents.groupby('stay_id')['value'].sum().reset_index().rename(columns={'value': 'total_urine_out_ml'})
proc_sum = procedureevents.groupby('stay_id')['itemid'].count().reset_index().rename(columns={'itemid': 'proc_count'})
date_sum = datetimeevents.groupby('stay_id')['itemid'].count().reset_index().rename(columns={'itemid': 'date_event_count'})
ingred_sum = ingredientevents.groupby('stay_id')['amount'].sum().reset_index().rename(columns={'amount': 'total_ingred_ml'})

# 3. THE "MAXIMUM EXTRACTION" SEQUENTIAL JOIN
# We use how='outer' to ensure NO ROW is left behind
# We start with icu_stays but keep everything from subsequent tables
master_icu = pd.merge(icu_stays, vitals_sum, on='stay_id', how='outer')
master_icu = pd.merge(master_icu, in_sum, on='stay_id', how='outer')
master_icu = pd.merge(master_icu, out_sum, on='stay_id', how='outer')
master_icu = pd.merge(master_icu, proc_sum, on='stay_id', how='outer')
master_icu = pd.merge(master_icu, date_sum, on='stay_id', how='outer')
master_icu = pd.merge(master_icu, ingred_sum, on='stay_id', how='outer')

# 4. Final Output
print(f"Maximum Extraction Complete.")
print(f"Total rows preserved: {len(master_icu)}")
print(f"Total features collected: {master_icu.shape[1]}")

# View the master datasheet
pd.set_option('display.max_columns', None)
master_icu.head()

Maximum Extraction Complete.
Total rows preserved: 140
Total features collected: 15


,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,icu_vitals_mean,icu_vitals_max,total_input_ml,total_urine_out_ml,proc_count,date_event_count,total_ingred_ml
0,10023117,28872262,30057454,Cardiac Vascular Intensive Care Unit (CVICU),Coronary Care Unit (CCU),2171-11-14 10:06:41,2171-11-18 20:49:43,4.446551,44.416088,251.0,10217.479889,11802.0,10.0,299.0,13830.021599
1,10032725,20611640,30101877,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2143-03-22 06:42:00,2143-03-25 15:05:33,3.349687,54.395508,2000.0,3316.333314,5069.0,7.0,55.0,2797.666619
2,10016742,27568122,30425410,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2178-07-22 08:19:00,2178-07-25 16:42:43,3.349803,69.267471,2700.0,7894.762545,3029.0,4.0,43.0,35818.392327
3,10031757,28477280,30458995,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2137-10-12 22:44:57,2137-10-14 17:08:34,1.766400,44.750359,860.0,9835.922232,4215.0,4.0,31.0,16270.844465
4,10022281,29642388,30585761,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2125-06-17 04:12:54,2125-06-18 14:55:55,1.446539,44.861905,239.0,2119.765445,2332.0,3.0,19.0,3849.611259
